In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.research_config import ResearchConfig
from src.cointegration import compute_returns, correlation_matrix, generate_candidate_pairs, screen_cointegration


# 02 Pair Selection
Generate correlation candidates from formation returns and apply fixed-orientation Engle-Granger screening with the I(1) filter.


In [ ]:
cfg = ResearchConfig()


In [ ]:
train_prices = pd.read_parquet("train_prices.parquet")
returns = compute_returns(train_prices)
correlations = correlation_matrix(returns)
candidate_pairs = generate_candidate_pairs(correlations, cfg.correlation_neighbors)
print(f"{len(candidate_pairs):,} candidate pairs")


In [ ]:
selected, spreads, results = screen_cointegration(
    train_prices,
    candidate_pairs,
    significance=cfg.cointegration_alpha,
    integration_alpha=cfg.integration_alpha,
)
results.to_parquet("cointegration_results.parquet")
selected.to_parquet("cointegrated_pairs.parquet")
formation_spreads = pd.DataFrame({f"{a}-{b}": values for (a, b), values in spreads.items()})
formation_spreads.to_parquet("formation_spreads.parquet")

print(f"{len(selected)} pairs retained")
display(results[["pair", "pvalue", "integration_screen", "selected"]].head(20))
display(selected[["pair", "alpha", "beta", "pvalue"]].head(10))
formation_spreads.iloc[:, :3].plot(figsize=(10, 4), title="Selected formation spreads")
plt.show()
